In [ ]:
model_ckpt = "google/vivit-b-16x2-kinetics400"
batch_size = 4
from transformers import VivitConfig, VivitModel

# Initializing a ViViT google/vivit-b-16x2-kinetics400 style configuration
configuration = VivitConfig()

# Initializing a model (with random weights) from the google/vivit-b-16x2-kinetics400 style configuration
model = VivitModel(configuration)

# Accessing the model configuration
configuration = model.config

print(configuration)

In [ ]:
# model_ckpt="facebook/timesformer-base-finetuned-k400"
# #model_ckpt = "MCG-NJU/videomae-base" # pre-trained model from which to fine-tune
# #model_ckpt = "google/vivit-b-16x2-kinetics400"
# batch_size = 4 # batch size for training and evaluation
# from transformers import TimesformerConfig, TimesformerModel

# # Initializing a TimeSformer timesformer-base style configuration
# configuration = TimesformerConfig()

# # Initializing a model from the configuration
# model = TimesformerModel(configuration)

# # Accessing the model configuration
# configuration = model.config
# print(configuration)


In [ ]:
import pathlib

# output_root_path = pathlib.Path(r'/media/cse/HDD/Shawon/shawon/MY DATA/WLASL Dataset')

output_root_path = pathlib.Path(r'/media/cse/HDD/Shawon/shawon/MY DATA/wlasl_100_class')

# Count videos in each set
video_count_train = len(list(output_root_path.glob("train/*/*.mp4")))
video_count_val = len(list(output_root_path.glob("val/*/*.mp4")))
video_count_test = len(list(output_root_path.glob("test/*/*.mp4")))

# Calculate total videos
video_total = video_count_train + video_count_val + video_count_test
print(f"Total videos: {video_total}")

# List all video file paths
all_video_file_paths = (
    list(output_root_path.glob("train/*/*.mp4")) +
    list(output_root_path.glob("val/*/*.mp4")) +
    list(output_root_path.glob("test/*/*.mp4"))
)

# Display the first five video file paths
print(all_video_file_paths[:5])

# Print total number of videos in each set and the first 5 video file paths for training
print(f"Total videos: {video_total}")
print(f"Training videos: {video_count_train}, Validation videos: {video_count_val}, Test videos: {video_count_test}")
# Adjust class label extraction logic
class_labels = sorted({str(path.parent.name) for path in all_video_file_paths})  # Use parent folder name as class label
label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}
# Print the unique class labels and their mappings
print(f"Unique classes: {list(label2id.keys())}.")
print(f"Label to ID mapping: {label2id}")
#print(f"ID to Label mapping: {id2label}")
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification
from transformers import VivitImageProcessor, VivitForVideoClassification
from transformers import AutoImageProcessor, TimesformerForVideoClassification

image_processor = VivitImageProcessor.from_pretrained("google/vivit-b-16x2-kinetics400")
model = VivitForVideoClassification.from_pretrained(
    model_ckpt,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,  # provide this in case you're planning to fine-tune an already fine-tuned checkpoint
)

# image_processor = AutoImageProcessor.from_pretrained("facebook/timesformer-base-finetuned-k400")
# model = TimesformerForVideoClassification.from_pretrained(
#     model_ckpt,
#     label2id=label2id,
#     id2label=id2label,
#     ignore_mismatched_sizes=True,)
# image_processor = AutoImageProcessor.from_pretrained("Shawon16/VideoMAE_BdSLW401_20_epochs_p5_SR_10")
# model = AutoModelForVideoClassification.from_pretrained(
#     model_ckpt,
#     label2id=label2id,
#     id2label=id2label,
#     ignore_mismatched_sizes=True,  # provide this in case you're planning to fine-tune an already fine-tuned checkpoint
# )

# image_processor = VideoMAEImageProcessor.from_pretrained(model_ckpt)
# model = VideoMAEForVideoClassification.from_pretrained(
#     model_ckpt,
#     label2id=label2id,
#     id2label=id2label,
#     ignore_mismatched_sizes=True,  # provide this in case you're planning to fine-tune an already fine-tuned checkpoint
# )

import pytorchvideo.data

from pytorchvideo.transforms import (
    ApplyTransformToKey,
    Normalize,
    RandomShortSideScale,
    RemoveKey,
    ShortSideScale,
    UniformTemporalSubsample,
)

from torchvision.transforms import (
    Compose,
    Lambda,
    RandomCrop,
    RandomHorizontalFlip,
    Resize,
)
import os
import os
import cv2  # Import OpenCV to handle video files
from pytorchvideo.data import make_clip_sampler, Ucf101
from torchvision.transforms import Compose
from transformers import VideoMAEForVideoClassification, VideoMAEConfig
from transformers import VivitConfig, VivitModel, VivitImageProcessor, VivitForVideoClassification
from transformers import AutoImageProcessor, TimesformerForVideoClassification


# Load the model configuration
#model_ckpt = "MCG-NJU/videomae-base"
#model_ckpt = "MCG-NJU/videomae-base-finetuned-kinetics"
#config = AutoModelForVideoClassification.from_pretrained(model_ckpt)
#model_ckpt = "facebook/timesformer-base-finetuned-k400"
#config = TimesformerForVideoClassification.from_pretrained(model_ckpt)

model_ckpt = "google/vivit-b-16x2-kinetics400"
# config = VivitForVideoClassification.from_pretrained(model_ckpt)
#num_frames_to_sample = config.num_frames


mean = image_processor.image_mean
std = image_processor.image_std
if "shortest_edge" in image_processor.size:
    height = width = image_processor.size["shortest_edge"]
else:
    height = image_processor.size["height"]
    width = image_processor.size["width"]
resize_to = (height, width)

num_frames_to_sample = model.config.num_frames
print(f"Number of frames to sample: {num_frames_to_sample}")
sample_rate = 6
fps = 60
clip_duration = num_frames_to_sample * sample_rate / fps

# Training dataset transformations.
train_transform = Compose(
    [
        ApplyTransformToKey(
            key="video",
            transform=Compose(
                [
                    UniformTemporalSubsample(num_frames_to_sample),
                    Lambda(lambda x: x / 255.0),
                    Normalize(mean, std),
                    Resize(resize_to),
                    RandomShortSideScale(min_size=256, max_size=320),
                    RandomCrop(resize_to),
                    RandomHorizontalFlip(p=0.5),
                    
                ]
            ),
        ),
    ]
)

# Training dataset.
train_dataset = pytorchvideo.data.Ucf101(
    data_path=os.path.join(output_root_path, "train"),
    clip_sampler=pytorchvideo.data.make_clip_sampler("random", clip_duration),
    decode_audio=False,
    transform=train_transform,
)



# Validation and evaluation datasets' transformations.
val_transform = Compose(
    [
        ApplyTransformToKey(
            key="video",
            transform=Compose(
                [
                    UniformTemporalSubsample(num_frames_to_sample),
                    Lambda(lambda x: x / 255.0),
                    Normalize(mean, std),
                    Resize(resize_to),
                ]
            ),
        ),
    ]
)

# Validation and evaluation datasets.
val_dataset = pytorchvideo.data.Ucf101(
    data_path=os.path.join(output_root_path, "val"),
    clip_sampler=pytorchvideo.data.make_clip_sampler("uniform", clip_duration),
    decode_audio=False,
    transform=val_transform,
)

test_dataset = pytorchvideo.data.Ucf101(
    data_path=os.path.join(output_root_path, "test"),
    clip_sampler=pytorchvideo.data.make_clip_sampler("uniform", clip_duration),
    decode_audio=False,
    transform=val_transform,
)
train_dataset.num_videos, val_dataset.num_videos, test_dataset.num_videos
# train data sample
sample_video = next(iter(train_dataset))
sample_video.keys()
def investigate_video(sample_video):
    """Utility to investigate the keys present in a single video sample."""
    for k in sample_video:
        if k == "video":
            print(k, sample_video["video"].shape)
        else:
            print(k, sample_video[k])

    print(f"Video label: {id2label[sample_video[k]]}")

investigate_video(sample_video)
import imageio
import numpy as np
from IPython.display import Image


def unnormalize_img(img):
    """Un-normalizes the image pixels."""
    img = (img * std) + mean
    img = (img * 255).astype("uint8")
    return img.clip(0, 255)


def create_gif(video_tensor, filename="sample.gif"):
    """Prepares a GIF from a video tensor.
    
    The video tensor is expected to have the following shape:
    (num_frames, num_channels, height, width).
    """
    frames = []
    for video_frame in video_tensor:
        frame_unnormalized = unnormalize_img(video_frame.permute(1, 2, 0).numpy())
        frames.append(frame_unnormalized)
    kargs = {"duration": 0.25}
    imageio.mimsave(filename, frames, "GIF", **kargs)
    return filename


def display_gif(video_tensor, gif_name="sample.gif"):
    """Prepares and displays a GIF from a video tensor."""
    video_tensor = video_tensor.permute(1, 0, 2, 3)
    gif_filename = create_gif(video_tensor, gif_name)
    return Image(filename=gif_filename)

video_tensor = sample_video["video"]
display_gif(video_tensor)
#print(video_tensor.shape)
import evaluate
metric = evaluate.load("accuracy")
import torch

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions."""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)


def collate_fn(examples):
    """The collation function to be used by `Trainer` to prepare data batches."""
    # permute to (num_frames, num_channels, height, width)
    pixel_values = torch.stack(
        [example["video"].permute(1, 0, 2, 3) for example in examples]
    )
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}
from huggingface_hub import login, hf_hub_download
# RnW token

# Authenticate using the API token
login(token=hf_token)

# WLASL

In [ ]:
import os
import json
from pathlib import Path
import cv2
import torch
import numpy as np
import evaluate
import pytorchvideo.data
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from scipy.stats import spearmanr
from statsmodels.nonparametric.smoothers_lowess import lowess
from transformers import (
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    VideoMAEImageProcessor,
    VideoMAEForVideoClassification,
)
# Correlation
from scipy.stats import spearmanr, pearsonr
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
# ---------- UPDATED compute_confusion_matrix ----------
from transformers import Trainer
from sklearn.metrics import confusion_matrix as sk_confusion_matrix
# ------------------------
# Determinism / reproducibility
# ------------------------

torch.multiprocessing.set_start_method("spawn", force=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# -------------------------
# Basic config
# -------------------------
metric = evaluate.load("accuracy")
num_epochs = 20
batch_size = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = f"/media/cse/HDD/Shawon/shawon/10 fold timesformer/ViViT_lsa64_coR"

os.makedirs(output_dir, exist_ok=True)

train_dir = os.path.join(output_root_path, "train")
val_dir = os.path.join(output_root_path, "val")
test_dir = os.path.join(output_root_path, "test")

# sanity-check dataset folders (used for frequency computations only)
if not os.path.isdir(train_dir):
    raise RuntimeError(f"train_dir does not exist: {train_dir}")
if not os.path.isdir(val_dir):
    print(f"Warning: val_dir does not exist: {val_dir} (only train frequencies require folder structure)")

# Get class labels from train_dir (preserves ordering)
class_labels = [d for d in sorted(os.listdir(train_dir)) if os.path.isdir(os.path.join(train_dir, d))]
if len(class_labels) == 0:
    raise RuntimeError(f"No class folders found under train_dir: {train_dir}")

# label mappings
label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}


def collate_fn(examples):
    """The collation function to be used by `Trainer` to prepare data batches."""
    # permute to (num_frames, num_channels, height, width)
    pixel_values = torch.stack(
        [example["video"].permute(1, 0, 2, 3) for example in examples]
    )
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}


# =========================
# Model setup
# =========================
# image_processor = VideoMAEImageProcessor.from_pretrained(model_ckpt)
# model = VideoMAEForVideoClassification.from_pretrained(
#     model_ckpt, label2id=label2id, id2label=id2label, ignore_mismatched_sizes=True
# ).to(device)

# image_processor = AutoImageProcessor.from_pretrained("facebook/timesformer-base-finetuned-k400")
# model = TimesformerForVideoClassification.from_pretrained(
#     model_ckpt,
#     label2id=label2id,
#     id2label=id2label,
#     ignore_mismatched_sizes=True,).to(device)
image_processor = VivitImageProcessor.from_pretrained("google/vivit-b-16x2-kinetics400")
model = VivitForVideoClassification.from_pretrained(
    model_ckpt,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,  # provide this in case you're planning to fine-tune an already fine-tuned checkpoint
)

# =========================
# Compute metrics & confusion
# =========================
def compute_confusion_matrix_and_output(trainer: Trainer, dataset, class_labels):
    """
    Runs trainer.predict once and returns:
      - PredictionOutput (so callers can reuse predictions/filenames)
      - derived metrics (overall_accuracy, precision, recall, f1, conf_mat, per_class_results, macro_results, top_results)
    """
    output = trainer.predict(dataset) # returning prediction from here to the for loop
    logits = output.predictions
    if logits is None:
        raise RuntimeError("No predictions returned by trainer.predict. Check dataset / trainer.")
    preds = np.argmax(logits, axis=1)
    labels = output.label_ids

    overall_accuracy = float((preds == labels).mean())

    conf_mat = sk_confusion_matrix(labels, preds, labels=np.arange(len(class_labels))).astype(np.int64)

    row_sums = conf_mat.sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        per_class_accuracy_arr = np.divide(np.diag(conf_mat), row_sums, where=row_sums != 0)

    per_class_accuracy = {class_labels[i]: (float(per_class_accuracy_arr[i]) if row_sums[i] != 0 else np.nan)
                          for i in range(len(class_labels))}

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=np.arange(len(class_labels)), average=None, zero_division=0
    )

    per_class_results = {
        "accuracy_per_class": per_class_accuracy,
        "precision": {class_labels[i]: float(precision[i]) for i in range(len(class_labels))},
        "recall": {class_labels[i]: float(recall[i]) for i in range(len(class_labels))},
        "f1": {class_labels[i]: float(f1[i]) for i in range(len(class_labels))},
        "support": {class_labels[i]: int(row_sums[i]) for i in range(len(class_labels))}
    }

    valid_accs = [v for v in per_class_accuracy_arr if not np.isnan(v)]
    macro_accuracy = float(np.mean(valid_accs)) if len(valid_accs) > 0 else 0.0

    try:
        macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
            labels, preds, average="macro", zero_division=0
        )
    except Exception:
        macro_precision = macro_recall = macro_f1 = 0.0

    macro_results = {
        "macro_accuracy": macro_accuracy,
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }

    num_classes = logits.shape[1]
    def topk_acc(k):
        k = min(k, num_classes)
        topk = np.argsort(logits, axis=1)[:, -k:]
        return float(np.mean([1 if label in row else 0 for label, row in zip(labels, topk)]))

    top_results = {
        "top1_accuracy": overall_accuracy,
        "top5_accuracy": topk_acc(5),
        "top10_accuracy": topk_acc(10),
    }

    return output, (overall_accuracy, precision, recall, f1, conf_mat,
                    per_class_results, macro_results, top_results)

# =========================
# Confusion Matrix Plot (Top-K)
# =========================
def plot_confusion_matrix(conf_matrix, class_labels, mode, overall_accuracy=None, macro_results=None,
                          output_dir=None, top_k=50, class_freqs=None):
    """
    Plot confusion matrix for top K most frequent classes in a PLOS ONE friendly layout.
    Uses raw counts. If class_freqs provided, selects top-K by frequency.
    """
    os.makedirs(output_dir, exist_ok=True)
    matplotlib.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 10,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

    # choose top-K classes by frequency if provided
    if class_freqs is not None:
        freq_counter = Counter(class_freqs)
        sorted_classes = sorted(class_labels, key=lambda c: freq_counter.get(c, 0), reverse=True)
        top_classes = sorted_classes[:top_k]# ---------- Head/Mid/Tail ----------
        indices = [class_labels.index(c) for c in top_classes]
        cm = conf_matrix[np.ix_(indices, indices)]
    else:
        top_k = min(top_k, len(class_labels))
        top_classes = class_labels[:top_k]
        cm = conf_matrix[:top_k, :top_k]

    # figure size (PLOS style)
    plos_width_inch = 180 / 25.4
    fig_height = plos_width_inch * 0.75
    plt.figure(figsize=(plos_width_inch, fig_height))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=top_classes,
        yticklabels=top_classes,
        cbar_kws={"shrink": 0.7},
        annot_kws={"size": 7.5},
        linewidths=0.3,
        linecolor="gray"
    )
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.xlabel("Predicted Label", fontsize=8, labelpad=5)
    plt.ylabel("True Label", fontsize=8, labelpad=5)

    if overall_accuracy is not None and macro_results is not None:
        metrics_line = (
            f"Acc: {overall_accuracy:.3f}, "
            f"Macro-Acc: {macro_results['macro_accuracy']:.3f}, "
            f"P: {macro_results['macro_precision']:.3f}, "
            f"R: {macro_results['macro_recall']:.3f}, "
            f"F1: {macro_results['macro_f1']:.3f}"
        )

    plt.title(f"ViViT LSA64 | {mode.upper()} | Top-{len(top_classes)} Classes\n{metrics_line}",
              fontsize=9, weight="bold", pad=10)
    plt.tight_layout(pad=0.8)

    save_path = os.path.join(output_dir, f"confusion_matrix_top{len(top_classes)}_{mode}.tiff")
    plt.savefig(save_path, format="tiff", dpi=300, bbox_inches="tight", pad_inches=0.1)
    plt.close()
    print(f"✅ Saved top-{len(top_classes)} confusion matrix to: {save_path}")

# =========================
# Long-tail plotting & correlation
def plot_long_tail_performance_vs_class_frequency(
    class_labels,
    per_class_results,
    data_dir,               # <-- FIXED: now accepts train/val/test directory
    mode,
    output_dir,
    add_trendline=None,
    add_linear_trend=True,
    add_smooth_trend=True,
    jitter_points=True,
    smooth_frac=0.35,
):
    if add_trendline is not None:
        add_linear_trend = add_trendline
        add_smooth_trend = add_trendline

    import os
    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    from scipy.stats import spearmanr
    from statsmodels.nonparametric.smoothers_lowess import lowess

    os.makedirs(output_dir, exist_ok=True)

    matplotlib.rcParams.update({
        "font.family": ["Times New Roman", "serif"],
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 10,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

    acc_dict = per_class_results.get("accuracy_per_class", {})
    f1_dict  = per_class_results.get("f1", {})

    # -----------------------------------
    # FIXED: frequency now depends on data_dir
    # -----------------------------------
    freq = {}
    for c in class_labels:
        folder = os.path.join(data_dir, c)
        if os.path.isdir(folder):
            try:
                freq[c] = sum(
                    os.path.isfile(os.path.join(folder, f))
                    for f in os.listdir(folder)
                )
            except:
                freq[c] = 0
        else:
            freq[c] = 0

    df = pd.DataFrame({
        "class": class_labels,
        "freq": [freq[c] for c in class_labels],
        "accuracy": [acc_dict.get(c, np.nan) for c in class_labels],
        "f1": [f1_dict.get(c, np.nan) for c in class_labels],
    }).sort_values("freq", ascending=False).reset_index(drop=True)

    # same quantile grouping
    df["group"] = pd.qcut(
        df["freq"].rank(method="first"),
        3,
        labels=["Tail", "Middle", "Head"]
    )

    group_means = df.groupby("group")[["accuracy", "f1"]].mean()
    print("\n" + "="*60)
    print(f"[{mode.upper()}] Head/Middle/Tail Means:")
    print(group_means.to_string())
    print("="*60)

    # ---------------------------
    # Spearman correlations
    # ---------------------------
    r_acc, p_acc = spearmanr(df["freq"], df["accuracy"], nan_policy="omit")
    r_f1,  p_f1  = spearmanr(df["freq"], df["f1"],       nan_policy="omit")

    plos_width = 180 / 25.4
    fig_height = plos_width * 0.75

    def make_plot(metric, ylabel, r, p, fname):
        plt.figure(figsize=(plos_width, fig_height))

        for g, sub in df.groupby("group"):
            x = sub["freq"].values
            y = sub[metric].values

            if jitter_points:
                x = x * (1 + np.random.normal(0, 0.02, size=len(x)))

            plt.scatter(x, y, alpha=0.8, s=25, label=f"{g} (n={len(sub)})")

        plt.xscale("log")
        plt.xlabel("Class Frequency (log scale)")
        plt.ylabel(ylabel)

        valid = df[["freq", metric]].dropna()
        valid = valid[valid["freq"] > 0]

        if len(valid) > 3:
            x_raw = valid["freq"].values
            x_log = np.log10(x_raw)
            y = valid[metric].values

            if add_linear_trend:
                A = np.polyfit(x_log, y, 1)
                xx = np.linspace(x_log.min(), x_log.max(), 300)
                yy = A[0]*xx + A[1]
                plt.plot(10**xx, yy, lw=1, color="black", alpha=0.9, label="Linear trend")

            if add_smooth_trend:
                sm = lowess(y, x_raw, frac=smooth_frac, return_sorted=True)
                plt.plot(sm[:,0], sm[:,1], lw=1, alpha=0.9, color="gray", label="LOWESS")

        plt.title(
            f"{ylabel} vs Frequency ({mode.upper()})\n"
            f"Spearman r={r:.3f}, p={p:.3f}",
            fontsize=9, weight="bold",
        )
        plt.grid(alpha=0.3)
        plt.legend(frameon=False)
        plt.tight_layout()

        plt.savefig(os.path.join(output_dir, fname),
                    dpi=300, format="tiff", pil_kwargs={"compression": "tiff_lzw"}, bbox_inches="tight")
        plt.close()

    make_plot("accuracy", "Class Accuracy", r_acc, p_acc, f"{mode}_longtail_accuracy.tiff")
    make_plot("f1",       "Class F1-Score", r_f1,  p_f1,  f"{mode}_longtail_f1.tiff")

    df.to_csv(os.path.join(output_dir, f"{mode}_longtail_data.csv"), index=False)

    return r_acc, p_acc, r_f1, p_f1, df, group_means

# avg frame vs acc code part
def frames_in_video(video_path):
    import cv2
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return total_frames

def compute_avg_frames_per_class(root_dir, class_labels, max_workers=8): # train_dir
    avg_frames = {}
    for cls in class_labels:
        folder = os.path.join(root_dir, cls)
        if not os.path.isdir(folder):
            avg_frames[cls] = 0
            continue
        vids = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))]
        if len(vids) == 0:
            avg_frames[cls] = 0
            continue
        frames_list = []
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = {ex.submit(frames_in_video, p): p for p in vids}
            for fut in as_completed(futures):
                frames_list.append(fut.result())
        avg_frames[cls] = float(np.mean([f for f in frames_list if f > 0])) if len([f for f in frames_list if f > 0])>0 else 0.0
    return avg_frames


# ---------------------------------------------------------
# 1) Load WLASL JSON → build fast video_id → signer_id map
# ---------------------------------------------------------
def build_wlasl_videoid_to_signerid(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    vid_to_signer = {}
    for gloss_entry in data:
        for inst in gloss_entry["instances"]:
            vid = str(inst.get("video_id"))
            signer = inst.get("signer_id")
            if vid is not None and signer is not None:
                vid_to_signer[vid] = signer

    print(f"✔ Loaded {len(vid_to_signer)} video_id → signer_id mappings.")
    return vid_to_signer


# ---------------------------------------------------------
# 2) Extract signer from filename using video_id
# ---------------------------------------------------------
def extract_signer_wlasl(filename, vid_to_signer):
    video_id = Path(filename).stem
    signer = vid_to_signer.get(video_id)
    if signer is None:
        print(f"⚠ Warning: video_id {video_id} not found in JSON")
    return signer


# ---------------------------------------------------------
# 3) Compute signer-wise stats
# ---------------------------------------------------------
def get_signer_wise_stats(y_true, y_pred, filenames, vid_to_signer):
    signer_stats = {}
    for true, pred, fname in zip(y_true, y_pred, filenames):
        signer = extract_signer_wlasl(fname, vid_to_signer)
        if signer not in signer_stats:
            signer_stats[signer] = {"correct": 0, "total": 0}
        signer_stats[signer]["total"] += 1
        if true == pred:
            signer_stats[signer]["correct"] += 1
    return signer_stats


# ---------------------------------------------------------
# 4) Save CSV + plot with dynamic x-axis labels (~20 labels)
# ---------------------------------------------------------


def save_signer_accuracy(signer_stats, out_dir, mode, num_labels=20):
    import os
    import pandas as pd
    import matplotlib.pyplot as plt

    # --------------------
    # 1. Build dataframe
    # --------------------
    rows = []
    for signer, stats in signer_stats.items():
        acc = stats["correct"] / stats["total"] if stats["total"] else 0
        rows.append({
            "signer": signer,
            "accuracy": acc,
            "total_samples": stats["total"],
            "correct_predictions": stats["correct"]
        })

    df_signer = pd.DataFrame(rows).sort_values("signer")
    csv_path = os.path.join(out_dir, f"per_signer_results_{mode}.csv")
    df_signer.to_csv(csv_path, index=False)
    print(f"📁 Saved per-signer CSV: {csv_path}")

    # --------------------
    # 2. Dynamic xtick spacing
    # --------------------
    total_signers = len(df_signer)
    step = max(1, total_signers // num_labels)
    label_indices = list(range(0, total_signers, step))
    label_signers = df_signer["signer"].iloc[label_indices]

    # --------------------
    # 3. PLOS ONE Figure Size Rules
    # --------------------
    # Required pixel range:
    # width: 789–2250 px
    # height: ≤2625 px
    # dpi = 300 ⇒ width_inches = px / 300

    dpi = 300
    min_px = 789
    max_px = 2250
    max_height_px = 2625

    # dynamic width scaling
    base_width_in = max(8, 0.12 * total_signers)

    # convert to pixels
    width_px = base_width_in * dpi

    # clamp to required range
    width_px = max(min_px, min(width_px, max_px))
    width_in = width_px / dpi

    # fixed safe height (below max)
    height_in = min(5.5, max_height_px / dpi)

    print(f"📐 Figure size (inches): {width_in:.2f} x {height_in:.2f}")
    print(f"📏 PLOS pixel size: {width_px:.0f} x {height_in*dpi:.0f}")

    plt.figure(figsize=(width_in, height_in))

    # --------------------
    # 4. Plotting
    # --------------------
    plt.scatter(df_signer["signer"], df_signer["accuracy"], s=35)
    plt.xticks(
        ticks=label_signers,
        labels=label_signers.astype(str),
        rotation=45,
        fontsize=10
    )
    plt.yticks(fontsize=10)
    plt.xlabel("Signer ID", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    plt.title(f"Accuracy vs Signer ({mode.upper()})", fontsize=12)
    plt.grid(alpha=0.3)
    plt.tight_layout()

    # --------------------
    # 5. PLOS ONE TIFF saving (LZW compression)
    # --------------------
    plot_path = os.path.join(out_dir, f"accuracy_vs_signer_{mode}.tiff")
    plt.savefig(
        plot_path,
        dpi=dpi,
        format="tiff",
        pil_kwargs={"compression": "tiff_lzw"}  # ensures <10MB
    )
    plt.close()

    print(f"📊 Saved PLOS ONE–compliant TIFF figure: {plot_path}")

    return df_signer


# ---------------------------------------------------------
# 5) Compute Unique Signers Per Class + Correlation + Plot
# ---------------------------------------------------------
def compute_unique_signers_per_class(y_true, filenames, vid_to_signer, class_labels):
    class_to_signers = {cls: set() for cls in class_labels}

    for true_label, fname in zip(y_true, filenames):
        signer = extract_signer_wlasl(fname, vid_to_signer)
        cls = class_labels[true_label]
        if signer is not None:
            class_to_signers[cls].add(signer)

    # convert to int counts
    return {cls: len(s) for cls, s in class_to_signers.items()}


def save_unique_signer_class_plot(class_signer_map, per_class_results, out_dir, mode):
    rows = []
    for cls, num_signers in class_signer_map.items():
        acc = per_class_results["accuracy_per_class"].get(cls, np.nan)
        rows.append({
            "class": cls,
            "unique_signers": num_signers,
            "accuracy": acc
        })

    df = pd.DataFrame(rows)

    # -------------------- Save CSV --------------------
    csv_path = os.path.join(out_dir, f"unique_signers_vs_accuracy_{mode}.csv")
    df.to_csv(csv_path, index=False)
    print(f"📁 Saved class signer-count CSV: {csv_path}")

    # -------------------- Correlations --------------------
    x = df["unique_signers"].values
    y = df["accuracy"].values

    spear_r, spear_p = spearmanr(x, y)
    pear_r, pear_p = pearsonr(x, y)

    print(f"📈 Spearman r={spear_r:.4f}, p={spear_p:.4g}")
    print(f"📈 Pearson  r={pear_r:.4f}, p={pear_p:.4g}")

    # -------------------- PLOS ONE FIGURE CONSTRAINTS --------------------
    dpi = 300
    min_px = 789
    max_px = 2250
    max_height_px = 2625

    num_classes = len(df)

    # base width scaling
    base_width_in = max(8, 0.09 * num_classes)
    width_px = base_width_in * dpi
    width_px = max(min_px, min(width_px, max_px))  # clamp
    width_in = width_px / dpi

    height_in = min(5.5, max_height_px / dpi)

    print(f"📐 Figure size (inches): {width_in:.2f} x {height_in:.2f}")
    print(f"📏 Pixel size: {width_px:.0f} x {height_in*dpi:.0f}")

    plt.figure(figsize=(width_in, height_in))

    # -------------------- Scatter plot --------------------
    jitter_x = x + np.random.normal(0, 0.15, size=len(x))
    plt.scatter(jitter_x, y, s=35, alpha=0.7)

    # -------------------- Regression line --------------------
    coeffs = np.polyfit(x, y, 1)
    poly = np.poly1d(coeffs)
    xs = np.linspace(min(x), max(x), 200)
    plt.plot(xs, poly(xs), color="black", linewidth=1.8)

    # -------------------- Labels & title --------------------
    plt.xlabel("Number of Unique Signers per Class", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    plt.title(f"Accuracy vs Unique Signers per Class ({mode.upper()})", fontsize=12)

    # -------------------- Corr text --------------------
    plt.text(
        0.02, 0.03,
        f"Spearman r={spear_r:.3f}, p={spear_p:.3g}\n"
        f"Pearson  r={pear_r:.3f}, p={pear_p:.3g}",
        transform=plt.gca().transAxes,
        fontsize=9,
        verticalalignment="bottom",
        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none")
    )

    plt.grid(alpha=0.3)
    plt.tight_layout()

    # -------------------- Save TIFF (LZW compression) --------------------
    plot_path = os.path.join(out_dir, f"accuracy_vs_unique_signers_{mode}.tiff")
    plt.savefig(
        plot_path,
        dpi=dpi,
        format="tiff",
        pil_kwargs={"compression": "tiff_lzw"}
    )
    plt.close()

    print(f"📊 Saved PLOS ONE–compliant plot: {plot_path}")

    return df

def compute_unique_signers_per_class_adjacent(y_true, filenames, vid_to_signer, class_labels):
    """
    For each class C:
       unique_signers = signers(C) - signers(adjacent_classes)
    Adjacent classes = previous and next class index.
    """
    # First gather signers per class
    class_to_signers = {cls: set() for cls in class_labels}

    for true_label, fname in zip(y_true, filenames):
        signer = extract_signer_wlasl(fname, vid_to_signer)
        cls = class_labels[true_label]
        if signer is not None:
            class_to_signers[cls].add(signer)

    # Now compute unique signers relative to adjacent classes
    unique_adjacent_map = {}

    num_classes = len(class_labels)

    for idx, cls in enumerate(class_labels):
        current = class_to_signers[cls]

        # adjacent classes: previous + next (if exist)
        adjacent_signers = set()

        if idx - 1 >= 0:
            prev_cls = class_labels[idx - 1]
            adjacent_signers |= class_to_signers[prev_cls]

        if idx + 1 < num_classes:
            next_cls = class_labels[idx + 1]
            adjacent_signers |= class_to_signers[next_cls]

        # unique = those appearing in class, but NOT in any adjacent class
        unique_signers = current - adjacent_signers

        unique_adjacent_map[cls] = len(unique_signers)

    return unique_adjacent_map
def build_anova_table(y_true, y_pred, filenames, vid_to_signer, class_labels):
    rows = []
    for true, pred, fname in zip(y_true, y_pred, filenames):
        signer = extract_signer_wlasl(fname, vid_to_signer)
        cls = class_labels[true]
        correct = 1 if true == pred else 0
        rows.append({
            "signer": signer,
            "class": cls,
            "correct": correct
        })
    return pd.DataFrame(rows)


from scipy.stats import f_oneway
def anova_signer_effect(df):
    groups = [group["correct"].values for _, group in df.groupby("signer")]

    F, p = f_oneway(*groups)
    print("📊 One-way ANOVA (accuracy ~ signer)")
    print(f"    F = {F:.4f},  p = {p:.6f}")

    if p < 0.05:
        print("👉 Signer variation has a statistically significant effect on accuracy.")
    else:
        print("👉 No significant effect of signer variation on accuracy.")


torch.cuda.empty_cache()


# Training arguments and Trainer
#batch = 2, grad_acc = 4
max_steps = (train_dataset.num_videos // (2 * 4)) * num_epochs if hasattr(train_dataset, "num_videos") else -1
args = TrainingArguments(
    output_dir=str(output_dir), remove_unused_columns=False,
    save_strategy="epoch", evaluation_strategy="epoch",
    learning_rate=5e-5, per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size, gradient_accumulation_steps=4,
    warmup_ratio=0.1, logging_strategy="epoch", max_steps=max_steps if max_steps>0 else None,
    load_best_model_at_end=True, metric_for_best_model="accuracy",
    num_train_epochs=num_epochs, report_to=["none"], fp16=True,
    weight_decay=0.01, push_to_hub=True,
)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": float(metric.compute(predictions=preds, references=labels)["accuracy"])}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

# =========================
# Train
# =========================
print("Training started (Fixed Split)")
trainer.train()

trainer.save_state()
trainer.save_model()
print("✅ Model and state saved locally.")

# =========================
# Evaluate train, val, test (single predict per dataset; reuse results)
# =========================video torch.Size([3, 16, 224, 224])

results = {}
avg_frames_map = {
    "train": compute_avg_frames_per_class(train_dir, class_labels),
    "val":   compute_avg_frames_per_class(val_dir, class_labels),
    "test":  compute_avg_frames_per_class(test_dir, class_labels)
}
print("✔ Done: Average frame counts computed once per split.")

datasets = [("train", train_dataset, train_dir), ("val", val_dataset, val_dir), ("test", test_dataset, test_dir)]

# plot conf mat
train_freq_list = []
for cls in class_labels:
    cls_folder = os.path.join(train_dir, cls)
    if os.path.isdir(cls_folder):
        n_files = len([f for f in os.listdir(cls_folder) if os.path.isfile(os.path.join(cls_folder, f))])
    else:
        n_files = 0
    train_freq_list += [cls] * n_files

# =========================
# Evaluate train / val / test and save metrics
# =========================
results_summary = []
JSON_PATH = "MY DATA/wlasl_100_class/WLASL_v0.3.json"
# Load video_id → signer_id mapping
vid_to_signer = build_wlasl_videoid_to_signerid(JSON_PATH)
# ---------- SAFE filename extractor ----------
def safe_get_video_name(sample):
    # Try several common keys, fallback to None
    if isinstance(sample, dict):
        for key in ("video_name", "filename", "path", "video_id", "video"):
            v = sample.get(key)
            if v is not None:
                return v
    # HuggingFace Dataset format: sample can be dict with 'video' as dict containing 'path'
    if hasattr(sample, "__getitem__"):
        try:
            if "video" in sample and isinstance(sample["video"], dict) and "path" in sample["video"]:
                return sample["video"]["path"]
        except Exception:
            pass
    return None
# ---------------------------------------------------------
# 5) MAIN PIPELINE — Integrate with your existing datasets loop
# ---------------------------------------------------------
for mode, dataset, data_dir in datasets:
    print(f"\nPredicting on {mode} dataset...")

    # single predict (reused)
    pred_output, metrics_tuple = compute_confusion_matrix_and_output(trainer, dataset, class_labels)
    acc, prec, rec, f1, cm, per_class_results, macro_results, top_results = metrics_tuple

    # long-tail analysis (uses class frequencies computed from the appropriate split)
    spearman_acc_r, spearman_acc_p, spearman_f1_r, spearman_f1_p, longtail_df, group_means = \
        plot_long_tail_performance_vs_class_frequency(
            class_labels,
            per_class_results,
            data_dir,
            mode,
            output_dir,
            add_trendline=True
        )

    # confusion matrix plot (uses train freqs for ordering if desired)
    plot_confusion_matrix(
        cm,
        class_labels,
        mode,
        overall_accuracy=acc,
        macro_results=macro_results,
        output_dir=output_dir,
        top_k=50,
        class_freqs=train_freq_list
    )

    # Reuse prediction outputs to compute signer stats (avoid extra predict)
    logits = pred_output.predictions
    y_pred = np.argmax(logits, axis=1)
    y_true = pred_output.label_ids

    # Obtain filenames robustly
    filenames = []
    for s in dataset:
        name = safe_get_video_name(s)
        filenames.append(name)

    signer_stats = get_signer_wise_stats(y_true, y_pred, filenames, vid_to_signer)
    save_signer_accuracy(signer_stats, output_dir, mode)

    # --------------------------------------------
    # Unique signers per class → correlation plot
    # --------------------------------------------
    # class_signer_map = compute_unique_signers_per_class(
    #     y_true, filenames, vid_to_signer, class_labels
    # )
    # class_signer_map = compute_unique_signers_per_class_adjacent(
    # y_true, filenames, vid_to_signer, class_labels
    # )


    # save_unique_signer_class_plot(
    #     class_signer_map,
    #     per_class_results,
    #     output_dir,
    #     mode
    # )
    class_signer_map = compute_unique_signers_per_class(
        y_true, filenames, vid_to_signer, class_labels
    )
   
    class_signer_map_adj = compute_unique_signers_per_class_adjacent(
        y_true, filenames, vid_to_signer, class_labels
    )
    # ---- create BOTH plots with unique mode names ----
    save_unique_signer_class_plot(
        class_signer_map,
        per_class_results,
        output_dir,
        mode
    )
    save_unique_signer_class_plot(
        class_signer_map_adj,
        per_class_results,
        output_dir,
        mode=f"{mode}_adjacent"
    )
    #ANOVA Test
    df_anova = build_anova_table(y_true, y_pred, filenames, vid_to_signer, class_labels)

    anova_signer_effect(df_anova)
 

    # Save per-class CSV with avg frames (avg_frames_map computed earlier)
    avg_frames = avg_frames_map.get(mode, {})
    rows = []
    for c in class_labels:
        rows.append({
            "class": c,
            "accuracy": per_class_results["accuracy_per_class"].get(c, np.nan),
            "precision": per_class_results["precision"].get(c, np.nan),
            "recall": per_class_results["recall"].get(c, np.nan),
            "f1": per_class_results["f1"].get(c, np.nan),
            "support": per_class_results["support"].get(c, 0),
            "avg_frames": avg_frames.get(c, 0),
        })
    per_class_path = os.path.join(output_dir, f"per_class_results_{mode}.csv")
    pd.DataFrame(rows).to_csv(per_class_path, index=False)
    print(f"✅ {mode.upper()} per-class results saved to: {per_class_path}")

    # ------------- 1. Create dataframe -------------
    df = pd.DataFrame(rows)

    # ------------- 2. Correlation calculations -------------
    from scipy.stats import spearmanr, pearsonr

    x = df["avg_frames"].values
    y = df["accuracy"].values

    spearman_r, spearman_p = spearmanr(x, y)
    pearson_r, pearson_p = pearsonr(x, y)

    print(f"📈 Spearman correlation (acc vs avg_frames): r={spearman_r:.4f}, p={spearman_p:.4g}")
    print(f"📈 Pearson correlation  (acc vs avg_frames): r={pearson_r:.4f}, p={pearson_p:.4g}")

    # ------------- 3. PLOS ONE FIGURE SIZE CONSTRAINTS -------------
    dpi = 300
    min_px = 789
    max_px = 2250
    max_height_px = 2625

    num_classes = df["class"].nunique()

    # Wider figure for more classes
    base_width_in = max(8, 0.10 * num_classes)
    width_px = base_width_in * dpi
    width_px = max(min_px, min(width_px, max_px))  # clamp
    width_in = width_px / dpi

    height_in = min(5.5, max_height_px / dpi)

    print(f"📐 Figure size (inches): {width_in:.2f} x {height_in:.2f}")
    print(f"📏 PLOS pixel size: {width_px:.0f} x {height_in*dpi:.0f}")

    plt.figure(figsize=(width_in, height_in))

    # ------------- 4. Scatter Plot -------------
    class_ids = pd.factorize(df["class"])[0]

    # Jitter x-axis to avoid overlapping when many points
    jitter_x = df["avg_frames"] + np.random.normal(0, 0.8, size=len(df))

    plt.scatter(
        jitter_x,
        df["accuracy"],
        c=class_ids,
        cmap="tab20",
        s=8,
        alpha=0.7,
        edgecolor="none"
    )

    # ------------- 5. Regression Line -------------
    coeffs = np.polyfit(df["avg_frames"], df["accuracy"], 1)
    poly_fn = np.poly1d(coeffs)
    x_sorted = np.sort(df["avg_frames"])
    plt.plot(x_sorted, poly_fn(x_sorted), color="black", linewidth=1.5, alpha=0.8)

    # ------------- 6. Labels / Title -------------
    plt.xlabel("Average Frames per Class")
    plt.ylabel("Accuracy")
    plt.title(f"Accuracy vs Average Frames ({mode.upper()})")

    # ------------- 7. On-plot correlation text -------------
    plt.text(
        0.02, 0.02,
        f"Spearman r={spearman_r:.3f}, p={spearman_p:.3g}\n"
        f"Pearson  r={pearson_r:.3f}, p={pearson_p:.3g}",
        transform=plt.gca().transAxes,
        fontsize=10,
        va="bottom",
        ha="left",
        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none")
    )

    plt.grid(True, alpha=0.3)
    cbar = plt.colorbar()
    cbar.set_label("Class Index")

    plt.tight_layout()

    # ------------- 8. PLOS ONE TIFF SAVING (LZW compression) -------------
    plot_path = os.path.join(output_dir, f"accuracy_vs_avg_frames_{mode}.tiff")

    plt.savefig(
        plot_path,
        dpi=dpi,
        format="tiff",
        pil_kwargs={"compression": "tiff_lzw"}  # ensures <10MB
    )
    plt.close()

    print(f"📊 Saved PLOS ONE–compliant TIFF: {plot_path}")


    # append summary
    results_summary.append({
        "mode": mode.upper(),
        "macro_accuracy": macro_results["macro_accuracy"],
        "macro_precision": macro_results["macro_precision"],
        "macro_recall": macro_results["macro_recall"],
        "macro_f1": macro_results["macro_f1"],
        "top1_accuracy": top_results["top1_accuracy"],
        "top5_accuracy": top_results["top5_accuracy"],
        "top10_accuracy": top_results["top10_accuracy"],
        "spearman_acc_r": spearman_acc_r,
        "spearman_acc_p": spearman_acc_p,
        "spearman_f1_r": spearman_f1_r,
        "spearman_f1_p": spearman_f1_p,
    })

summary_df = pd.DataFrame(results_summary)
summary_path = os.path.join(output_dir, "overall_results_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\n📊 All metrics summary saved to: {summary_path}\n")
print("=== Final Summary ===")
print(summary_df.to_string(index=False))
trainer.push_to_hub()
